In [1]:
import os
import pandas as pd

In [2]:
df = pd.read_csv('diversed-data.csv')
df.iloc[0:14].values

array([['[CLS] \\frac{22}{12} \\, - \\, 4\\,\\frac{6}{15} \\,=\\, 3\\,\\frac{7}{27} [SEP] The problem is 22/12 - 4 6/15. The student response is 3 7/27. Likely a conversion issue: bundling whole and fractional parts, then operating across them during subtraction. [SEP]',
        'conv-a',
        '\\frac{22}{12} \\, - \\, 4\\,\\frac{6}{15} \\,=\\, 3\\,\\frac{7}{27}',
        'The problem is 22/12 - 4 6/15. The student response is 3 7/27. Likely a conversion issue: bundling whole and fractional parts, then operating across them during subtraction.',
        'SUB FRAC 22 12 ADD INT 4 FRAC 6 15', '3\\,\\frac{7}{27}',
        '\\frac{22}{12} \\, - \\, 4\\,\\frac{6}{15} \\,=\\, 3\\,\\frac{7}{27}',
        'ADD INT 3 FRAC 7 27',
        'SUB FRAC 22 12 ADD INT 4 FRAC 6 15 EQ ADD INT 3 FRAC 7 27',
        '22/12 - 4 6/15'],
       ['[CLS] \\frac{22}{12} \\, - \\, 4\\,\\frac{6}{15} \\,=\\, 8\\,\\frac{1}{15} [SEP] The problem is 22/12 - 4 6/15. The student response is 8 1/15. Likely a conversio

In [3]:
df.columns

Index(['input', 'label', 'formula_latex', 'context_text', 'problem_opt',
       'answer_latex', 'prob_ans_latex', 'answer_opt', 'prob_ans_opt',
       'problem_raw'],
      dtype='object')

In [4]:
df.iloc[0].to_dict()

{'input': '[CLS] \\frac{22}{12} \\, - \\, 4\\,\\frac{6}{15} \\,=\\, 3\\,\\frac{7}{27} [SEP] The problem is 22/12 - 4 6/15. The student response is 3 7/27. Likely a conversion issue: bundling whole and fractional parts, then operating across them during subtraction. [SEP]',
 'label': 'conv-a',
 'formula_latex': '\\frac{22}{12} \\, - \\, 4\\,\\frac{6}{15} \\,=\\, 3\\,\\frac{7}{27}',
 'context_text': 'The problem is 22/12 - 4 6/15. The student response is 3 7/27. Likely a conversion issue: bundling whole and fractional parts, then operating across them during subtraction.',
 'problem_opt': 'SUB FRAC 22 12 ADD INT 4 FRAC 6 15',
 'answer_latex': '3\\,\\frac{7}{27}',
 'prob_ans_latex': '\\frac{22}{12} \\, - \\, 4\\,\\frac{6}{15} \\,=\\, 3\\,\\frac{7}{27}',
 'answer_opt': 'ADD INT 3 FRAC 7 27',
 'prob_ans_opt': 'SUB FRAC 22 12 ADD INT 4 FRAC 6 15 EQ ADD INT 3 FRAC 7 27',
 'problem_raw': '22/12 - 4 6/15'}

# 1. Tokenization

## 1.1. Install and Imports

In [5]:
# If running locally and you don't have transformers:
# !pip install -q transformers accelerate torch --upgrade

import pandas as pd
import numpy as np
from typing import Optional, Dict, List, Tuple
from collections import Counter

import torch
from transformers import AutoTokenizer, DataCollatorWithPadding

/Users/ankitmishra/Desktop/Desktop-Ankit MacBookPro(2)/Ankit/Desktop/Research/Lexi&Kenichi/MathBERT/math_bert/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1.2. Load Dataset

In [6]:
CSV_PATH = "diversed-data.csv"   
df = pd.read_csv(CSV_PATH)
df.head(3)

,input,label,formula_latex,context_text,problem_opt,answer_latex,prob_ans_latex,answer_opt,prob_ans_opt,problem_raw
0,"[CLS] \frac{22}{12} \, - \, 4\,\frac{6}{15} \,...",conv-a,"\frac{22}{12} \, - \, 4\,\frac{6}{15} \,=\, 3\...",The problem is 22/12 - 4 6/15. The student res...,SUB FRAC 22 12 ADD INT 4 FRAC 6 15,"3\,\frac{7}{27}","\frac{22}{12} \, - \, 4\,\frac{6}{15} \,=\, 3\...",ADD INT 3 FRAC 7 27,SUB FRAC 22 12 ADD INT 4 FRAC 6 15 EQ ADD INT ...,22/12 - 4 6/15
1,"[CLS] \frac{22}{12} \, - \, 4\,\frac{6}{15} \,...",conv-b,"\frac{22}{12} \, - \, 4\,\frac{6}{15} \,=\, 8\...",The problem is 22/12 - 4 6/15. The student res...,SUB FRAC 22 12 ADD INT 4 FRAC 6 15,"8\,\frac{1}{15}","\frac{22}{12} \, - \, 4\,\frac{6}{15} \,=\, 8\...",ADD INT 8 FRAC 1 15,SUB FRAC 22 12 ADD INT 4 FRAC 6 15 EQ ADD INT ...,22/12 - 4 6/15
2,"[CLS] \frac{22}{12} \, - \, 4\,\frac{6}{15} \,...",conv-c,"\frac{22}{12} \, - \, 4\,\frac{6}{15} \,=\, 7\...",The problem is 22/12 - 4 6/15. The student res...,SUB FRAC 22 12 ADD INT 4 FRAC 6 15,"7\,\frac{1}{3}","\frac{22}{12} \, - \, 4\,\frac{6}{15} \,=\, 7\...",ADD INT 7 FRAC 1 3,SUB FRAC 22 12 ADD INT 4 FRAC 6 15 EQ ADD INT ...,22/12 - 4 6/15


## 1.3.Choose which OPT to append

In [7]:
USE_OPT = True
OPT_FIELD = "prob_ans_opt"   # or "problem_opt" if you prefer
OPT_PREFIX = " OPT: "        # plain text prefix that will appear in segment B
MAX_LEN = 256               # keep 512 for debug; later you can reduce if needed

LABELS = [
    "conv-a","conv-b","conv-c","conv-d",
    "po-a","po-b","po-c","po-d",
    "pf-a","pf-b","pf-c","pf-d","pf-e",
    "a",
]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

## 1.4. Build segment A (LaTeX) and segment B (context [+OPT])

In [8]:
def build_context_with_opt(context: str, opt_text: Optional[str], prefix: str = " OPT: ") -> str:
    if opt_text and str(opt_text).strip():
        return f"{context.strip()}{prefix}{opt_text.strip()}"
    return context.strip()

formula = df["formula_latex"].astype(str).tolist()
if USE_OPT:
    if OPT_FIELD not in df.columns:
        raise ValueError(f"Requested OPT field '{OPT_FIELD}' not found in CSV.")
    context = [
        build_context_with_opt(c, o, OPT_PREFIX)
        for c, o in zip(df["context_text"].astype(str), df[OPT_FIELD].astype(str))
    ]
else:
    context = df["context_text"].astype(str).tolist()

labels = df["label"].map(label2id).astype(int).tolist()
len(formula), len(context), len(labels)

(30000, 30000, 30000)

## 1.5. Initialize the MathBERT Tokenizer

In [9]:
TOKENIZER_NAME = "tbs17/MathBERT"
tok = AutoTokenizer.from_pretrained(TOKENIZER_NAME, use_fast=True)
tok

BertTokenizerFast(name_or_path='tbs17/MathBERT', vocab_size=30522, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=False, added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}
)

## 1.6. Tokenize a single row and inspect

In [10]:
ROW = 0  # change to inspect different rows

lhs = formula[ROW]
rhs = context[ROW]

enc = tok(
    lhs, rhs,
    add_special_tokens=True,          # inserts [CLS] lhs [SEP] rhs [SEP]
    truncation="longest_first",
    max_length=MAX_LEN,
    return_attention_mask=True,
    return_token_type_ids=True,       # 0 for lhs, 1 for rhs
    return_special_tokens_mask=True,
    return_overflowing_tokens=True,   # notify if truncation happened
    return_length=True,
)

print("Label:", df.loc[ROW, "label"], "| id:", labels[ROW])
print("\n--- Segment A (LaTeX) ---\n", lhs)
print("\n--- Segment B (context [+OPT]) ---\n", rhs)
print("\nLengths:", enc["length"], "| overflow tokens:", len(enc.get("overflowing_tokens", [])))

Label: conv-a | id: 0

--- Segment A (LaTeX) ---
 \frac{22}{12} \, - \, 4\,\frac{6}{15} \,=\, 3\,\frac{7}{27}

--- Segment B (context [+OPT]) ---
 The problem is 22/12 - 4 6/15. The student response is 3 7/27. Likely a conversion issue: bundling whole and fractional parts, then operating across them during subtraction. OPT: SUB FRAC 22 12 ADD INT 4 FRAC 6 15 EQ ADD INT 3 FRAC 7 27

Lengths: [112] | overflow tokens: 0


In [11]:
def unwrap_first_sequence(enc):
    """
    HuggingFace returns list-of-lists when return_overflowing_tokens=True.
    This converts it to a single (first) sequence view so pretty printers work.
    """
    flat = dict(enc)
    seq_keys = ["input_ids", "attention_mask", "token_type_ids", "special_tokens_mask"]
    for k in seq_keys:
        if isinstance(flat.get(k), list) and len(flat[k]) > 0 and isinstance(flat[k][0], list):
            flat[k] = flat[k][0]
    # Some tokenizers also return 'length' as a list when overflowing
    if isinstance(flat.get("length"), list) and flat["length"]:
        flat["length"] = flat["length"][0]
    return flat

# Build with overflow info ON (so you can see if truncation happened)
enc = tok(
    lhs, rhs,
    add_special_tokens=True,
    truncation="longest_first",
    max_length=MAX_LEN,
    return_attention_mask=True,
    return_token_type_ids=True,
    return_special_tokens_mask=True,
    return_overflowing_tokens=True,   # keep it True
    return_length=True,
)

# Optional: display truncation info
if enc.get("overflowing_tokens"):
    print(f"WARNING: truncation occurred; dropped {len(enc['overflowing_tokens'][0])} tokens.")

# Unwrap a single view for pretty printing
enc_view = unwrap_first_sequence(enc)



## 1.7. Show a token table (see boundaries, UNKs, segments)

In [12]:
def token_table(tokenizer, enc, max_rows=220):
    ids  = enc["input_ids"]
    tti  = enc["token_type_ids"]
    attn = enc["attention_mask"]
    spec = enc["special_tokens_mask"]
    toks = tokenizer.convert_ids_to_tokens(ids)

    print(f"{'idx':>4} | {'token':<20} | {'id':<6} | seg | attn | special")
    print("-"*60)
    for i,(tk,ti,am,sp) in enumerate(zip(toks, ids, attn, spec)):
        mark = ""
        if tk == tokenizer.cls_token: mark = "<CLS>"
        elif tk == tokenizer.sep_token: mark = "<SEP>"
        elif ti == tokenizer.unk_token_id: mark = "<UNK>"
        row = f"{i:>4} | {tk:<20} | {ti:<6} |  {tti[i]}  |  {am}   |   {sp}  {mark}"
        print(row)
        if i+1 >= max_rows and len(ids) > max_rows:
            print(f"... ({len(ids)-max_rows} more)")
            break

token_table(tok, enc_view)

 idx | token                | id     | seg | attn | special
------------------------------------------------------------
   0 | [CLS]                | 101    |  0  |  1   |   1  <CLS>
   1 | \                    | 1032   |  0  |  1   |   0  
   2 | fra                  | 25312  |  0  |  1   |   0  
   3 | ##c                  | 2278   |  0  |  1   |   0  
   4 | {                    | 1063   |  0  |  1   |   0  
   5 | 22                   | 2570   |  0  |  1   |   0  
   6 | }                    | 1065   |  0  |  1   |   0  
   7 | {                    | 1063   |  0  |  1   |   0  
   8 | 12                   | 2260   |  0  |  1   |   0  
   9 | }                    | 1065   |  0  |  1   |   0  
  10 | \                    | 1032   |  0  |  1   |   0  
  11 | ,                    | 1010   |  0  |  1   |   0  
  12 | -                    | 1011   |  0  |  1   |   0  
  13 | \                    | 1032   |  0  |  1   |   0  
  14 | ,                    | 1010   |  0  |  1   |   0  
  15

## 1.8. Sanity checks + human-readable decode

In [13]:
def sanity_checks(tokenizer, enc):
    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"])
    # 1) [CLS] and 2 [SEP]
    assert toks[0] == tokenizer.cls_token, "First token isn't [CLS]"
    sep_positions = [i for i,t in enumerate(toks) if t == tokenizer.sep_token]
    assert len(sep_positions) == 2, f"Expected 2 [SEP], found {len(sep_positions)}"
    # 2) segment split
    a, b = sep_positions
    types = enc["token_type_ids"]
    assert set(types[1:a]) == {0}, "Segment A must be type 0"
    assert set(types[a+1:b]) == {1}, "Segment B must be type 1"
    # 3) UNK rate
    unk = sum(int(i == tokenizer.unk_token_id) for i in enc["input_ids"])
    if unk:
        print(f"NOTE: {unk} UNK tokens (~{100*unk/len(enc['input_ids']):.2f}%).")
    # 4) Truncation info
    if enc.get("overflowing_tokens"):
        print(f"WARNING: truncated {len(enc['overflowing_tokens'])} tokens.")

sanity_checks(tok, enc_view)

print("\n--- Decoded back (quick eyeball) ---")
print(tok.decode(enc_view["input_ids"]))


--- Decoded back (quick eyeball) ---
[CLS] \ frac { 22 } { 12 } \, - \, 4 \, \ frac { 6 } { 15 } \, = \, 3 \, \ frac { 7 } { 27 } [SEP] the problem is 22 / 12 - 4 6 / 15. the student response is 3 7 / 27. likely a conversion issue : bundling whole and fractional parts, then operating across them during subtraction. opt : sub frac 22 12 add int 4 frac 6 15 eq add int 3 frac 7 27 [SEP]


## 1.9. Dataset-wide stats (lengths, truncation, UNK)

In [14]:
def dataset_stats(formula, context, tokenizer, max_len=512, sample=None):
    idxs = range(len(formula)) if sample is None else range(min(sample, len(formula)))
    lengths, trunc_count, unk_total = [], 0, 0
    for i in idxs:
        enc = tokenizer(
            formula[i], context[i],
            add_special_tokens=True, truncation="longest_first",
            max_length=max_len, return_length=True
        )
        lengths.append(enc["length"])
        if enc["length"] == max_len: trunc_count += 1
        ids = tokenizer(
            formula[i], context[i],
            add_special_tokens=True, truncation="longest_first",
            max_length=max_len
        )["input_ids"]
        unk_total += sum(int(x == tokenizer.unk_token_id) for x in ids)

    print(f"Samples: {len(list(idxs))}")
    print(f"Length: min={min(lengths)}, mean={np.mean(lengths):.1f}, max={max(lengths)} (cap={max_len})")
    print(f"Truncated: {trunc_count} ({100*trunc_count/len(list(idxs)):.1f}%)")
    print(f"Avg UNK per example: {unk_total/len(list(idxs)):.2f}")

dataset_stats(formula, context, tok, max_len=MAX_LEN, sample=1000)  # adjust sample

Samples: 1000
Length: min=[87], mean=105.4, max=[129] (cap=256)
Truncated: 0 (0.0%)
Avg UNK per example: 0.00


## 1.10. Build a small dataloader and preview a batch

In [15]:
from torch.utils.data import Dataset, DataLoader

class PairDataset(Dataset):
    def __init__(self, formula, context, labels, tokenizer, max_len=512):
        self.A = formula; self.B = context; self.y = labels
        self.tok = tokenizer; self.max_len = max_len

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        enc = self.tok(
            self.A[i], self.B[i],
            add_special_tokens=True, truncation="longest_first",
            max_length=self.max_len, return_attention_mask=True,
            return_token_type_ids=True
        )
        return {
            "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "token_type_ids": torch.tensor(enc["token_type_ids"], dtype=torch.long),
            "labels": torch.tensor(self.y[i], dtype=torch.long),
        }

collator = DataCollatorWithPadding(tokenizer=tok)
dl = DataLoader(PairDataset(formula, context, labels, tok, MAX_LEN),
                batch_size=4, shuffle=False, collate_fn=collator)

batch = next(iter(dl))
{k: v.shape for k,v in batch.items()}

{'input_ids': torch.Size([4, 116]),
 'attention_mask': torch.Size([4, 116]),
 'token_type_ids': torch.Size([4, 116]),
 'labels': torch.Size([4])}

# 2. Classifier

## 2.1. Setup & Imports

In [16]:
# If needed:
# !pip install -q transformers accelerate torch scikit-learn --upgrade

import os, math, time, json, random
import numpy as np
import pandas as pd
from typing import Optional, List, Dict

import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import GradScaler, autocast
from sklearn.metrics import accuracy_score, f1_score, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
    DataCollatorWithPadding,
)

## 2.2. Config and labels

In [17]:
CSV_PATH   = "diversed-data.csv"      # your generated file
TOKENIZER  = "tbs17/MathBERT"
MODEL_NAME = "tbs17/MathBERT"
USE_OPT    = True                      # append OPT to context
OPT_FIELD  = "prob_ans_opt"            # use full equation tree
OPT_PREFIX = " OPT: "
MAX_LEN    = 512
BATCH_SIZE = 16
EPOCHS     = 3
LR         = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
GRAD_CLIP   = 1.0
SEED        = 42
SAVE_DIR    = "mathbert-error-classifier"

LABELS = [
    "conv-a","conv-b","conv-c","conv-d",
    "po-a","po-b","po-c","po-d",
    "pf-a","pf-b","pf-c","pf-d","pf-e",
    "a",
]
label2id = {l:i for i,l in enumerate(LABELS)}
id2label = {i:l for l,i in label2id.items()}

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
device = torch.device("mps" if torch.mps.is_available() else "cpu")
device

device(type='mps')

## 2.3. Load CSV and build splits (Stratified)

In [18]:
from sklearn.model_selection import train_test_split

df = pd.read_csv(CSV_PATH)
assert {"formula_latex","context_text","label"}.issubset(df.columns)

train_df, temp_df = train_test_split(
    df, test_size=0.20, random_state=SEED, stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED, stratify=temp_df["label"]
)

len(train_df), len(val_df), len(test_df)

(24000, 3000, 3000)

## 2.4. Dataset + DataLoader (pairwise encoding; OPT appended to context)

In [19]:
def build_context_with_opt(context: str, opt_text: Optional[str], prefix: str = " OPT: ") -> str:
    if USE_OPT and (OPT_FIELD in df.columns) and isinstance(opt_text, str) and opt_text.strip():
        return f"{context.strip()}{prefix}{opt_text.strip()}"
    return context.strip()

class PairDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, tokenizer, max_len=512):
        self.formula = frame["formula_latex"].astype(str).tolist()
        if USE_OPT:
            self.context = [
                build_context_with_opt(c, o, OPT_PREFIX)
                for c, o in zip(frame["context_text"].astype(str), frame.get(OPT_FIELD, [""]*len(frame)))
            ]
        else:
            self.context = frame["context_text"].astype(str).tolist()
        self.labels = frame["label"].map(label2id).astype(int).tolist()
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self): return len(self.labels)

    def __getitem__(self, i: int):
        enc = self.tok(
            self.formula[i], self.context[i],
            add_special_tokens=True,
            truncation="longest_first",
            max_length=self.max_len,
            return_attention_mask=True,
            return_token_type_ids=True,
        )
        return {
            "input_ids":      torch.tensor(enc["input_ids"], dtype=torch.long),
            "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            "token_type_ids": torch.tensor(enc["token_type_ids"], dtype=torch.long),
            "labels":         torch.tensor(self.labels[i], dtype=torch.long),
        }

tok = AutoTokenizer.from_pretrained(TOKENIZER, use_fast=True)

train_ds = PairDataset(train_df, tok, MAX_LEN)
val_ds   = PairDataset(val_df, tok, MAX_LEN)
test_ds  = PairDataset(test_df, tok, MAX_LEN)

collator = DataCollatorWithPadding(tokenizer=tok)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collator, pin_memory=torch.cuda.is_available())
val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator, pin_memory=torch.cuda.is_available())
test_dl  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator, pin_memory=torch.cuda.is_available())

len(train_dl), len(val_dl), len(test_dl)

(1500, 188, 188)

## 2.5. Model + Optimizer + Scheduler

In [20]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)
model.to(device)

# Params with/without weight decay
no_decay = ["bias", "LayerNorm.weight"]
optim_groups = [
    {"params": [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)], "weight_decay": WEIGHT_DECAY},
    {"params": [p for n,p in model.named_parameters() if any(nd in n for nd in no_decay)], "weight_decay": 0.0},
]
optimizer = AdamW(optim_groups, lr=LR)

# Scheduler
num_train_steps = len(train_dl) * EPOCHS
num_warmup = int(WARMUP_RATIO * num_train_steps)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup, num_train_steps)

scaler = GradScaler(enabled=torch.cuda.is_available())  # mixed precision

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at tbs17/MathBERT and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/var/folders/fj/xnqh3s0x18dfyk4wsqptqx840000gn/T/ipykernel_39184/303024470.py:22: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=torch.cuda.is_available())  # mixed precision


## 2.6. Train/eval loops with early stopping (macro-F1)

In [ ]:
best_f1 = -1.0
patience = 2
stale = 0

os.makedirs(SAVE_DIR, exist_ok=True)

for epoch in range(1, EPOCHS+1):
    model.train()
    running_loss = 0.0
    running_correct = 0
    running_examples = 0

    for step, batch in enumerate(train_dl, 1):
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad(set_to_none=True)
        with autocast(enabled=torch.cuda.is_available()):
            out = model(**batch)
            loss = out.loss

        # compute training accuracy for this batch (before backward)
        with torch.no_grad():
            preds = out.logits.argmax(dim=-1)
            running_correct += (preds == batch["labels"]).sum().item()
            running_examples += batch["labels"].size(0)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()

        if step % 50 == 0 or step == 1:
            train_acc = running_correct / max(1, running_examples)
            print(
                f"epoch {epoch} step {step}/{len(train_dl)} | "
                f"loss {running_loss/step:.4f} | "
                f"train_acc {train_acc:.4f}"
            )

    # Validation
    val_metrics, _ = evaluate(val_dl, model)
    print(
        f"epoch {epoch} | "
        f"val_loss {val_metrics['loss']:.4f} | "
        f"val_acc {val_metrics['acc']:.4f} | "
        f"val_f1 {val_metrics['f1_macro']:.4f}"
    )

    # Early stop on macro-F1
    if val_metrics["f1_macro"] > best_f1:
        best_f1 = val_metrics["f1_macro"]
        stale = 0
        model.save_pretrained(SAVE_DIR)
        tok.save_pretrained(SAVE_DIR)
        with open(os.path.join(SAVE_DIR, "label_maps.json"), "w") as f:
            json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)
        print(f"✅ saved new best to {SAVE_DIR} (f1_macro={best_f1:.4f})")
    else:
        stale += 1
        if stale > patience:
            print("⏹️ early stopping")
            break

/var/folders/fj/xnqh3s0x18dfyk4wsqptqx840000gn/T/ipykernel_39184/3216776375.py:17: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=torch.cuda.is_available()):


epoch 1 step 1/1500 | loss 2.7826 | train_acc 0.0000
epoch 1 step 50/1500 | loss 2.6698 | train_acc 0.0900
epoch 1 step 100/1500 | loss 2.4582 | train_acc 0.2981
epoch 1 step 150/1500 | loss 1.8939 | train_acc 0.5254
epoch 1 step 200/1500 | loss 1.4400 | train_acc 0.6441
epoch 1 step 250/1500 | loss 1.1568 | train_acc 0.7153
epoch 1 step 300/1500 | loss 0.9664 | train_acc 0.7627
epoch 1 step 350/1500 | loss 0.8299 | train_acc 0.7966
epoch 1 step 400/1500 | loss 0.7271 | train_acc 0.8220


## 2.7. Load best model and test evaluation

In [ ]:
best_model = AutoModelForSequenceClassification.from_pretrained(SAVE_DIR).to(device)
test_metrics, (y_pred, y_true) = evaluate(test_dl, best_model)
print("TEST:", test_metrics)
print("\nPer-class report:\n", classification_report(y_true, y_pred, target_names=LABELS, digits=4))

## 2.8. Quick inference on a single row

In [ ]:
row = 0  # pick any
f = df.loc[row, "formula_latex"]
c = df.loc[row, "context_text"]
if USE_OPT:
    c = f"{c}{OPT_PREFIX}{df.loc[row, OPT_FIELD]}"

pair = tok(
    f, c,
    add_special_tokens=True,
    truncation="longest_first",
    max_length=MAX_LEN,
    return_tensors="pt",
)
pair = {k:v.to(device) for k,v in pair.items()}

best_model.eval()
with torch.no_grad():
    logits = best_model(**pair).logits
probs = torch.softmax(logits, dim=-1).squeeze().cpu().numpy()

pred_id = int(np.argmax(probs))
print("Pred:", id2label[pred_id], "| True:", df.loc[row, "label"])
topk = np.argsort(-probs)[:5]
for k in topk:
    print(f"{id2label[k]:<6}  {probs[k]:.4f}")